# Nature Explorers — Base44 → Supabase Migration

**Prerequisites:**
```bash
pip install supabase pandas python-dotenv
```

Create `migration/.env`:
```
SUPABASE_URL=https://xxxx.supabase.co
SUPABASE_SERVICE_ROLE_KEY=your_service_role_key
```

CSV files expected in `migration/data/`:
- `Refuge_export.csv` → `refuges` table
- `MountainGuide_export.csv` → `mountain_guides` table
- `Organizer_export.csv` → `organizers` table
- `HikingTrip_export.csv` → `hiking_trips` table

**Run Cell 2 (Inspect) before any upload cell.**

In [ ]:
# Cell 1 — Setup
import pandas as pd
import numpy as np
import json
import os
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

if not SUPABASE_URL or not SUPABASE_SERVICE_ROLE_KEY:
    raise ValueError("Missing SUPABASE_URL or SUPABASE_SERVICE_ROLE_KEY in .env")

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

DATA = "data/"

def clean(df):
    """Replace pandas NaN/NaT with None so Supabase accepts the row."""
    return df.where(pd.notna(df), None)

def parse_json_col(val, default=None):
    """Parse a JSON string column safely; return default on empty/null."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return default
    s = str(val).strip()
    if s in ("", "nan", "null"):
        return default
    try:
        return json.loads(s)
    except Exception:
        return default

def upload(table, records, chunk=500):
    """Insert rows in batches; print errors without stopping execution."""
    total, errors = 0, 0
    for i in range(0, len(records), chunk):
        batch = records[i:i + chunk]
        res = supabase.table(table).insert(batch).execute()
        if hasattr(res, 'error') and res.error:
            print(f"  ✗ [{table}] batch {i // chunk}: {res.error}")
            errors += 1
        else:
            total += len(batch)
    print(f"  ✓ {total} rows inserted into '{table}' ({errors} batch errors)")

print("✓ Setup complete")

In [ ]:
# Cell 2 — Inspect CSVs  (READ-ONLY — no uploads)
# Prints column names and first 2 rows for every available CSV.

files = {
    "Refuge_export.csv"       : "refuges",
    "MountainGuide_export.csv": "mountain_guides",
    "Organizer_export.csv"    : "organizers",
    "HikingTrip_export.csv"   : "hiking_trips",
}

for filename, table in files.items():
    path = DATA + filename
    if not os.path.exists(path):
        print(f"⚠  {path} not found — skip")
        continue
    df = pd.read_csv(path, nrows=2)
    print(f"\n=== {filename} → {table} ({len(pd.read_csv(path))} rows) ===")
    print("Columns:", list(df.columns))

In [ ]:
# Cell 3 — Refuges
# CSV columns: google_maps_link, altitude, mountain, website, lng, facebook,
#              name, instagram, type, lat, refuge_link, capacity,
#              id, created_date, updated_date, created_by_id, created_by, is_sample

path = DATA + "Refuge_export.csv"
df = pd.read_csv(path)
print(f"Loaded {len(df)} refuges")

df = df.rename(columns={
    "altitude": "altitude_m",
    "lng"     : "longitude",
    "lat"     : "latitude",
})

# Keep only columns that exist in the Supabase refuges table
keep = [c for c in ["name", "altitude_m", "location", "description", "latitude", "longitude", "image_url"] if c in df.columns]
records = clean(df[keep]).to_dict("records")

upload("refuges", records)

In [ ]:
# Cell 4 — Mountain Guides
# CSV columns: profile_photo_url, full_name, user_id, cover_photo_url, bio,
#              organizer_codes, years_of_experience, certifications, is_verified,
#              social_media, status, id, created_date, updated_date, ...

path = DATA + "MountainGuide_export.csv"
df = pd.read_csv(path)
print(f"Loaded {len(df)} mountain guides")

df = df.rename(columns={
    "profile_photo_url": "profile_image_url",
})

# certifications is a JSON array in the CSV — parse back to list
if "certifications" in df.columns:
    df["certifications"] = df["certifications"].apply(lambda x: parse_json_col(x, default=[]))

keep = [c for c in ["full_name", "profile_image_url", "bio", "location", "certifications"] if c in df.columns]
records = clean(df[keep]).to_dict("records")

upload("mountain_guides", records)

In [ ]:
# Cell 5 — Organizers
# CSV columns: website, social_profiles, full_name, phone, organizer_code,
#              profile_picture_url, bio, years_of_experience, certifications,
#              is_verified, email, username, id, created_date, updated_date, ...

path = DATA + "Organizer_export.csv"
df = pd.read_csv(path)
print(f"Loaded {len(df)} organizers")

df = df.rename(columns={
    "profile_picture_url": "profile_image_url",
    "is_verified"        : "verified",
})

# Defaults for columns not present in the export
df["plan"] = "free"
if "verified" in df.columns:
    df["verified"] = df["verified"].apply(lambda x: bool(x) if pd.notna(x) else False)
else:
    df["verified"] = False

keep = [c for c in ["organizer_code", "full_name", "username", "bio",
                     "profile_image_url", "location", "verified",
                     "payment_instructions", "plan", "plan_expires_at"] if c in df.columns]
records = clean(df[keep]).to_dict("records")

upload("organizers", records)

In [ ]:
# Cell 6 — Hiking Trips
# CSV columns: end_date, booked_clicks, is_promoted, latitude, pricing_options,
#              description, departure_from, title, guide_id, cancel_policy, price,
#              start_date, longitude, requirements, distance_km, image_url,
#              organizer_code, total_slots, event_url, external_link, tags,
#              difficulty, elevation_gain_m, location, view_count, status, id, ...

path = DATA + "HikingTrip_export.csv"
df = pd.read_csv(path)
print(f"Loaded {len(df)} hiking trips")

df = df.rename(columns={
    "total_slots": "total_attendees",
})

# Parse JSON array columns (stored as strings in CSV)
for col in ["pricing_options", "tags", "requirements", "departure_from"]:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: parse_json_col(x, default=[]))

# Numeric defaults
df["price"]          = pd.to_numeric(df.get("price",          0), errors="coerce").fillna(0)
df["view_count"]     = pd.to_numeric(df.get("view_count",     0), errors="coerce").fillna(0).astype(int)
df["total_attendees"]= pd.to_numeric(df.get("total_attendees",10), errors="coerce").fillna(10).astype(int)
df["status"]         = df["status"].fillna("upcoming") if "status" in df.columns else "upcoming"

# Drop Base44-internal columns not in Supabase schema
drop_cols = ["id", "booked_clicks", "is_promoted", "guide_id", "external_link",
             "created_date", "updated_date", "created_by_id", "created_by", "is_sample"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

records = clean(df).to_dict("records")
# Remove None-valued keys so Supabase doesn't reject missing-column rows
records = [{k: v for k, v in row.items() if v is not None} for row in records]

upload("hiking_trips", records)

In [ ]:
# Cell 7 — Verification: row counts per table

tables = ["refuges", "mountain_guides", "organizers", "hiking_trips"]

print(f"{'Table':<20} {'Supabase rows':>15}")
print("-" * 37)
for t in tables:
    res = supabase.table(t).select("id", count="exact").limit(0).execute()
    count = res.count if hasattr(res, "count") and res.count is not None else len(res.data)
    print(f"{t:<20} {count:>15}")